# AeroFleet — AutoGraph + AutoRAG over a structured graph and a generated graph

This notebook deploys the **Arango AI Suite** end to end with the
[`arango-ai-sdk`](https://github.com/arangoml/arango-ai-sdk) and answers questions that need **two graphs**:

| Graph | Built by | Contents |
|---|---|---|
| **Structured fleet graph** (`fleetGraph`) | `pipeline/loader.py` from NASA C-MAPSS telemetry + synthetic operational data | 100 engines, aircraft, subsystems, sensors, parts (with stock), technicians, work orders |
| **Generated knowledge graph** (`<project>_kg`) | **AutoGraph**, from 36 maintenance documents in `corpus/initial/` | Entities and relations extracted from Airworthiness Directives, Service Bulletins and AMM task cards |

```
corpus/*.md ──upload──▶ AutoGraph ──▶ chunks · entities · relations (generated graph) ─┐
                                                                                         ├─▶ AutoRAG retriever ─▶ answers
pipeline/loader.py ─────────────────▶ engines · aircraft · parts · technicians (fleet) ──┘     (LOCAL / GLOBAL / UNIFIED,
                                                                                               custom tools, Deep Search)
```

| § | Section | What it shows |
|---|---|---|
| 0 | Configuration | Settings from the environment or prompts — no secrets stored in the notebook |
| 1 | Setup | Clone this repo, install it and the AI Suite SDK |
| 2 | Structured graph | Load, score and embed the fleet graph; one multi-hop traversal |
| 3 | AI Suite project | Project + secret profiles |
| 4 | Upload | Upload the maintenance corpus |
| 5 | AutoGraph | Deploy, build corpus graph, choose strategies, import, inspect the generated graph |
| 6 | AutoRAG | Deploy the retriever; LOCAL / GLOBAL / UNIFIED questions |
| 7 | Custom retriever | Tools over **both** graphs; Deep Search across them; partition check |
| 8 | Incremental update | Add a new emergency AD and watch the answers change |
| 9 | Cleanup | Undeploy services (optional) |

**Prerequisites:** a fresh Arango cluster with the AI Suite; this notebook running on the platform notebook server with **Python 3.12+** and outbound access to GitHub and NASA; an OpenAI API key; a database user AutoGraph can use for File Parser recovery.

The answer key for every question is in [`corpus/QUESTIONS.md`](../corpus/QUESTIONS.md).

## 0 · Configuration

Each setting is read from the environment; anything missing is prompted for. Passwords and API keys are read with `getpass` and only kept in this kernel's memory.

The AI Suite SDK cannot delete projects, so each run uses a fresh project name with a timestamp suffix. Set `PROJECT_NAME` before running to reuse an existing project.

In [ ]:
import datetime
import getpass
import os
import sys
from pathlib import Path

assert sys.version_info >= (3, 12), (
    f"arango-ai-sdk needs Python 3.12+, this kernel runs {sys.version.split()[0]}"
)


def cfg(name: str, default: str | None = None, secret: bool = False) -> str:
    """Return a setting from the environment, else prompt for it (secrets are not echoed)."""
    value = os.environ.get(name) or default
    if not value:
        value = getpass.getpass(f"{name}: ") if secret else input(f"{name}: ")
    os.environ[name] = value  # the repo's pipeline modules read the same env vars
    return value


# Deployment endpoint, port included, no trailing slash. The platform sets
# ARANGO_DEPLOYMENT_ENDPOINT on its notebook servers; otherwise you are prompted.
ARANGO_URL = cfg("ARANGO_URL", os.environ.get("ARANGO_DEPLOYMENT_ENDPOINT")).rstrip("/")
os.environ["ARANGO_URL"] = ARANGO_URL
ARANGO_DB = cfg("ARANGO_DB", "aerofleet_enablement")  # must contain "aerofleet"
ARANGO_USER = cfg("ARANGO_USER")
ARANGO_PASSWORD = cfg("ARANGO_PASSWORD", secret=True)
OPENAI_API_KEY = cfg("OPENAI_API_KEY", secret=True)
CHAT_MODEL = cfg("CHAT_MODEL", "gpt-4o")
# Must match the model used for engines.notesEmbedding (§2) so the retriever's
# query vectors and the stored vectors are comparable.
EMBEDDING_MODEL = cfg("EMBEDDING_MODEL", "text-embedding-3-small")
FPS_RECOVERY_USERNAME = cfg("FPS_RECOVERY_USERNAME", ARANGO_USER)
PROJECT = cfg("PROJECT_NAME", f"aerofleet_docs_{datetime.datetime.now():%m%d%H%M}")
CATEGORY = "maintenance_docs"  # second label of the upload scope [PROJECT, CATEGORY]

assert "aerofleet" in ARANGO_DB, "ARANGO_DB must contain 'aerofleet' (loader safety guard)"
print(f"endpoint : {ARANGO_URL}\ndatabase : {ARANGO_DB}\nuser     : {ARANGO_USER}")
print(f"project  : {PROJECT}\nmodels   : {CHAT_MODEL} / {EMBEDDING_MODEL}")

## 1 · Setup

Uses the repository this notebook sits in, or clones the `ai-suite-enablement` branch next to your home directory. Then installs the repo (pipeline + helpers) and the AI Suite SDK, pinned to a specific commit for reproducibility (the SDK is pre-alpha and not yet on public PyPI).

In [ ]:
import subprocess

REPO_URL = "https://github.com/mickgillesh/arango-predictive-maintenance-demo.git"
BRANCH = "ai-suite-enablement"
SDK_REF = "1cc8d2c83923e8583f76f58910c79b28a59513b3"  # arangoml/arango-ai-sdk main, 2026-10-16

here = Path.cwd()
REPO_DIR = next(
    (p for p in (here, *here.parents) if (p / "pipeline" / "loader.py").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path.home() / "arango-predictive-maintenance-demo"
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            check=True)
print("repo:", REPO_DIR)

In [ ]:
%pip install -q -e {REPO_DIR}
%pip install -q "arango-ai-sdk @ git+https://github.com/arangoml/arango-ai-sdk@{SDK_REF}"

In [ ]:
import importlib.metadata

# Work from the repo root and make its packages importable without a kernel restart.
os.chdir(REPO_DIR)
for p in (REPO_DIR, REPO_DIR / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("arango-ai-sdk", importlib.metadata.version("arango-ai-sdk"))
print("python-arango", importlib.metadata.version("python-arango"))

## 2 · Structured fleet graph

Creates the database if needed, then runs the same pipeline as `make reset` on a laptop:

1. **Load** — NASA C-MAPSS FD001 telemetry (`readings`) and a deterministic synthetic operational graph (`fleetGraph`). *Destructive:* drops and reloads the fleet collections; guarded to databases whose name contains `aerofleet`.
2. **Score** — a health index per engine from real sensor drift → predicted remaining useful life (RUL), risk bucket and **driver subsystems**.
3. **Embed** — a deterministic technician note per engine, embedded with `EMBEDDING_MODEL` into `engines.notesEmbedding` (used by a custom retriever tool in §7).

In [ ]:
from backend.db import ensure_database, get_db

print("created database" if ensure_database() else "database exists:", ARANGO_DB)

from pipeline import loader  # noqa: E402
from pipeline.embed_runner import embed_and_writeback  # noqa: E402
from pipeline.scorer_runner import score_and_writeback  # noqa: E402

loader.main()
print("risk distribution:", dict(score_and_writeback()))
embed_and_writeback()
db = get_db()

In [ ]:
from nb_helpers import peek

print({c: db.collection(c).count() for c in
       ["engines", "aircraft", "subsystems", "sensors", "parts", "technicians", "workOrders",
        "readings"]})

# Critical engines — compare with the table at the top of corpus/QUESTIONS.md
CRITICAL = peek(db, """
FOR e IN engines
  FILTER e.riskBucket == @risk
  LET ac = FIRST(FOR a IN 1..1 OUTBOUND e installedOn RETURN a)
  SORT e.predictedRUL
  RETURN {engine: e._key, model: e.model, rul: e.predictedRUL,
          driverSubsystems: e.driverSubsystems, tail: ac.tailNumber, base: ac.base}
""", {"risk": "critical"})
CRITICAL

**Talking point — one multi-hop query.** For the most critical engine: its aircraft, degrading subsystems, the parts those subsystems need (zero stock = *blocking*) and the technicians certified for them at the aircraft's base. This is the structured half of the answer the retriever will combine with documents in §7.

In [ ]:
from backend.aql import Q_ENGINE_IMPACT

worst = CRITICAL.iloc[0]["engine"]
impact = next(iter(db.aql.execute(Q_ENGINE_IMPACT, bind_vars={"engineId": worst})))
print("engine", worst, "on", impact["aircraft"])
print("degrading:", impact["degradingSubsystems"])
print("blocking parts:", [p["name"] for p in impact["blockingParts"]])
print("technicians at base:", [t["name"] for t in impact["technicians"]])

## 3 · AI Suite project and secrets

An AutoGraph **project** owns one AutoGraph service and any number of AutoRAG retrievers. LLM keys are stored once as **secret profiles** and referenced by id.

In [ ]:
from arango_ai_sdk import ArangoAIClient
from arango_ai_sdk.exceptions import ProjectNotFoundError

client = ArangoAIClient(base_url=ARANGO_URL, username=ARANGO_USER, password=ARANGO_PASSWORD,
                        db_name=ARANGO_DB)
try:
    project = client.project.get(project_name=PROJECT)
    print("reusing project", project.name)
except ProjectNotFoundError:
    project = client.project.create(
        project_name=PROJECT, project_type="autograph",
        description="AeroFleet engine maintenance documents — enablement session")
    print("created project", project.name, "in", project.db_name)

chat = client.secret_profile.create(name=f"{PROJECT}-chat", secret=OPENAI_API_KEY,
                                    provider="openai")
embed = client.secret_profile.create(name=f"{PROJECT}-embed", secret=OPENAI_API_KEY,
                                     provider="openai")
print("secret profiles:", chat.profile_id, embed.profile_id)

## 4 · Upload the maintenance corpus

36 Markdown documents for the four fleet engine models (plus a few realistic distractors for engines we don't operate). Every file is uploaded under the scope `[PROJECT, "maintenance_docs"]`; the second label is the **category** AutoGraph builds from.

The 37th document, `corpus/incremental/AD-EASA-2026-0214.md`, is **held back** for §8.

In [ ]:
import pandas as pd

from pipeline.corpus import CORPUS_DIR, load_corpus

docs = pd.DataFrame([d.__dict__ for d in load_corpus()])
print(docs.groupby(["stage", "doc_type"]).size().to_string(), "\n")
print("total words:", docs.words.sum())

results = client.files.upload_folder(CORPUS_DIR / "initial", scope=[PROJECT, CATEGORY])
failed = [r for r in results if r.status != "ok"]
print(f"uploaded {len(results) - len(failed)} of {len(results)}")
for r in failed:
    print(" ", r.name, r.status, r.detail)

## 5 · AutoGraph — from documents to a knowledge graph

Three steps, run separately here so each can be explained (`ingest()` does all three in one call):

1. **Corpus build** — extracts text, links similar documents and groups them into clusters.
2. **Strategize** — picks a strategy per cluster: **FullGraphRAG** (entities + relations + communities) or **VectorRAG** (chunks + embeddings). Each cluster becomes a **partition** with its own `rag_partition_id`.
3. **Orchestrate** — imports each partition into the database.

In [ ]:
deployed = project.autograph.deploy(
    chat_api_provider="openai", embedding_api_provider="openai",
    chat_secret_profile_id=chat.profile_id, embedding_secret_profile_id=embed.profile_id,
    fps_recovery_username=FPS_RECOVERY_USERNAME,
    chat_model=CHAT_MODEL, embedding_model=EMBEDDING_MODEL,
)
print("service:", deployed.service_info.service_id)
assert project.autograph.wait_until_ready(), "AutoGraph did not become ready"

In [ ]:
build = project.autograph.corpus_build(
    [CATEGORY], on_progress=lambda s: print("build", s.status, s.progress))
print("documents:", build.document_count, "| error:", build.error_code or "-")

strategize = project.autograph.strategize(
    [CATEGORY], on_progress=lambda s: print("strategize", s.status,
                                            f"{s.clusters_done}/{s.clusters_total}"))
stored = project.autograph.strategies()
print("strategies:", stored.total_strategies, dict(stored.strategy_type_counts))
pd.DataFrame([{"partition": s.rag_partition_id, "strategy": s.strategy_type,
               "documents": s.document_count, "entity_types": ", ".join(s.entity_types[:6])}
              for s in stored.strategies])

In [ ]:
orchestration = project.autograph.orchestrate(
    [CATEGORY], on_progress=lambda s: print("orchestrate", s.status, s.phase))
print("nothing stale" if orchestration is None else f"orchestration: {orchestration.status}")

### What AutoGraph created

The cells below list the new collections and check the edge types that the custom retriever tools in §7 rely on (`PART_OF`, `MENTIONED_IN`, `RELATED_TO`). Open the **Graph Visualizer** on the generated graph to explore it visually.

In [ ]:
from pipeline.retriever_tools import discover_autograph_collections

names = sorted(c["name"] for c in db.collections() if c["name"].startswith(PROJECT))
print({n: db.collection(n).count() for n in names})
print("graphs:", [g["name"] for g in db.graphs() if g["name"].startswith(PROJECT)])

AG = discover_autograph_collections([c["name"] for c in db.collections()], PROJECT)
print(AG)
peek(db, "FOR r IN @@rel COLLECT type = r.type WITH COUNT INTO n SORT n DESC RETURN {type, n}",
     {"@rel": AG.relations})

In [ ]:
# A sample of extracted entities (schema shown as AutoGraph wrote it)
if AG.entities:
    display(peek(db, "FOR e IN @@ents LIMIT 8 RETURN UNSET(e, '_rev', '_id', 'embedding')",
                 {"@ents": AG.entities}))
else:
    print("no Entities collection (VectorRAG-only partitions)")

## 6 · AutoRAG — questions over the documents

Deploy a retriever for the project and ask the document-only questions from `QUESTIONS.md` §A.

| Mode | How it retrieves |
|---|---|
| `LOCAL` | Starts from entities named in the question and follows their relationships |
| `GLOBAL` | Reads community summaries across the graph — themes and overviews |
| `UNIFIED` | Combines matching text passages with the graph |

In [ ]:
from arango_ai_sdk.models import QueryType

retriever = project.autorag.deploy(
    chat_api_provider="openai", embedding_api_provider="openai",
    chat_secret_profile_id=chat.profile_id, embedding_secret_profile_id=embed.profile_id,
    chat_model=CHAT_MODEL, embedding_model=EMBEDDING_MODEL,
)
rag = retriever.autorag
assert rag.wait_until_ready(), "retriever did not become ready"
RETRIEVER_URL = rag.service_url
print("retriever:", rag.service_id, RETRIEVER_URL)

In [ ]:
from nb_helpers import answer_mentions, show

A = {
    "A1": ("What does AD-EASA-2024-0312 require for CFM56-7B27 HPC blades?", QueryType.LOCAL),
    "A2": ("How can we tell HPC fouling from HPC erosion on a CFM56 engine?", QueryType.UNIFIED),
    "A3": ("Which documents apply to the V2527-A5 combustor, and how often must it be inspected?",
           QueryType.LOCAL),
    "A4": ("Which sensor indicates HPT cooling problems, and which tasks or directives use it?",
           QueryType.UNIFIED),
    "A5": ("What are the main deterioration mechanisms across our maintenance documentation?",
           QueryType.GLOBAL),
    "A6": ("Does AD-EASA-2023-0891 apply to any engine in our fleet?", QueryType.LOCAL),
}
answers = {}
for key, (question, mode) in A.items():
    answers[key] = rag.ask(question, query_type=mode)
    show(f"{key} [{mode.name}] {question}", answers[key])

In [ ]:
# Spot-check against the answer key (corpus/QUESTIONS.md)
print("A1", answer_mentions(answers["A1"], "100 cycles", "0.8 mm", "1.2 mm"))
print("A3", answer_mentions(answers["A3"], "AD-FAA-2024-0583", "200 cycles"))
print("A4", answer_mentions(answers["A4"], "W31"))
print("A6", answer_mentions(answers["A6"], "Trent 700"))

## 7 · Custom retriever — querying both graphs

AutoRAG's built-in modes only see the generated graph. **Custom Retriever tools** extend it to any collection in the database, including the structured fleet graph. Each tool is a document in a Tools collection: a collection to search (lexical / semantic / hybrid), an AQL template that expands the hits, and a natural-language description that lets **Deep Search** choose tools automatically.

| Tool | Graph | Search → expansion |
|---|---|---|
| `aerofleet_techdocs_v2` | generated | hybrid on chunks → source document |
| `aerofleet_kg_entities_v1` | generated | hybrid on entities → mentions + related entities |
| `aerofleet_fleet_engine_v2` | structured | hybrid on model + technician notes → aircraft, subsystems, parts, technicians, work orders |
| `aerofleet_fleet_status_v5` | structured | lexical on risk bucket → same expansion, critical/warning only |

The tool definitions live in `pipeline/retriever_tools.py`; the AQL in `backend/aql.py`. The SDK's `ask()` doesn't take `custom_tools`, so these queries call the retriever's REST endpoint (`nb_helpers.graphrag_query`).

In [ ]:
import json

from nb_helpers import CUSTOM, get_jwt, graphrag_query

from pipeline.retriever_tools import (
    FLEET_ENGINE_TOOL_ID,
    FLEET_STATUS_TOOL_ID,
    KG_TOOL_ID,
    TECHDOCS_TOOL_ID,
    build_tools,
    default_tools_collection,
    register_tools,
)

TOOLS = build_tools(AG)
TOOLS_COLLECTION = default_tools_collection(PROJECT)
print("registered in", TOOLS_COLLECTION, register_tools(db, TOOLS_COLLECTION, TOOLS))
print(json.dumps(TOOLS[-1], indent=1)[:1500], "…")


def jwt() -> str:
    return get_jwt(ARANGO_URL, ARANGO_USER, ARANGO_PASSWORD)


def ask_tool(tool_id: str, question: str, **kw) -> dict:
    """Run one custom tool (query_type CUSTOM); return the body, or {'error': ...}."""
    try:
        return graphrag_query(RETRIEVER_URL, jwt(), question, query_type=CUSTOM,
                              custom_tools=[tool_id], **kw)
    except RuntimeError as exc:
        return {"error": str(exc)}

**Check: which Tools collection does the retriever read?** The docs say `{project}_Tools`; an earlier deployment used `Tools`. If the document tool fails from the first, the cell registers the tools in the other and retries, and prints which one worked — note it for the session.

In [ ]:
probe = "What does AD-EASA-2024-0312 require?"
r = ask_tool(TECHDOCS_TOOL_ID, probe)
if "error" in r or not r.get("result"):
    alternative = "Tools" if TOOLS_COLLECTION != "Tools" else f"{PROJECT}_Tools"
    print(f"{TOOLS_COLLECTION} did not work ({r.get('error', 'empty result')[:200]}); "
          f"trying {alternative}")
    TOOLS_COLLECTION = alternative
    register_tools(db, TOOLS_COLLECTION, TOOLS)
    r = ask_tool(TECHDOCS_TOOL_ID, probe)
print("Tools collection in use:", TOOLS_COLLECTION, "| ok:", "error" not in r)
show(f"[{TECHDOCS_TOOL_ID}] {probe}", r)

**Check: is `partition_id` on engine documents needed?** An earlier version of this demo copied the AutoGraph partition id onto every engine so the retriever's automatic partition selection would not filter out structured results. This branch removes that workaround. The cell asks the fleet-status tool about critical engines and looks for their aircraft tail numbers (distinctive strings from the structured graph) in the answer. If none appear, it retries with `auto_select_partitions: false`.

In [ ]:
TAILS = list(CRITICAL["tail"])
B1 = "Which engines are critical and what is blocking their maintenance?"


def tails_found(resp: dict) -> list[str]:
    return [t for t in TAILS if t in (resp.get("result") or "")]


b1 = ask_tool(FLEET_STATUS_TOOL_ID, B1)
found = tails_found(b1)
if found:
    PARTITION_FLAGS = {}
    print("✔ structured results returned with default partition selection — no partition_id needed")
else:
    print("✘ no critical tails in the answer; retrying with auto_select_partitions=False")
    b1 = ask_tool(FLEET_STATUS_TOOL_ID, B1, auto_select_partitions=False)
    found = tails_found(b1)
    PARTITION_FLAGS = {"auto_select_partitions": False} if found else {}
    print("✔ works with auto_select_partitions=False" if found
          else "✘ still no structured results — STOP and investigate before the session")
print("tails found:", found)
show(f"[{FLEET_STATUS_TOOL_ID}] {B1}", b1)

In [ ]:
q = "Which CFM56-7B27 engines show HPC degradation in their technician notes?"
show(f"[{FLEET_ENGINE_TOOL_ID}] {q}", ask_tool(FLEET_ENGINE_TOOL_ID, q, **PARTITION_FLAGS))

if AG.entities:
    q = "How is HPC fouling related to EGT margin and compressor washing?"
    show(f"[{KG_TOOL_ID}] {q}", ask_tool(KG_TOOL_ID, q))

### Deep Search across both graphs

Deep Search (`query_type` CUSTOM + `use_llm_planner: true`) lets an LLM planner break the question into steps and pick tools by their descriptions — fleet tools for *which engines and which parts*, document tools for *which directives and tasks* — then synthesise one answer.

In [ ]:
ALL_TOOL_IDS = [t["tool_id"] for t in TOOLS]
B2 = ("For our critical engines, which airworthiness directives and maintenance tasks apply to "
      "their degrading subsystems, and are the required parts in stock?")


def deep_search(question: str) -> dict:
    return graphrag_query(RETRIEVER_URL, jwt(), question, query_type=CUSTOM,
                          use_llm_planner=True, custom_tools=ALL_TOOL_IDS,
                          **PARTITION_FLAGS)


b2 = deep_search(B2)
show(f"[Deep Search] {B2}", b2)
print(answer_mentions(b2, *TAILS, "AD-EASA-2024-0312", "AD-EASA-2023-1423", "HPC blade set"))

## 8 · Incremental update — a new emergency AD arrives

`corpus/incremental/AD-EASA-2026-0214.md` is an emergency directive that **supersedes AD-EASA-2024-0312**: CFM56-7B27 HPC stages 1-5 every **50 cycles**, within **10 cycles** for engines with an HPC trend alert, and replacement of the complete **HPC blade set stage 1-4** on any reject — a part with **zero stock** in the fleet graph.

1. Ask the question **before** the update.
2. Upload the new document to the same category and run `ingest(..., incremental=True)`: the existing corpus is kept, only new or changed documents are processed, and only stale partitions are re-imported.
3. Ask again and compare.

In [ ]:
C1 = "What are the current mandatory HPC inspection requirements for CFM56-7B27 engines?"
before = rag.ask(C1, query_type=QueryType.UNIFIED)
show(f"BEFORE — {C1}", before)
print(answer_mentions(before, "AD-EASA-2026-0214", "AD-EASA-2024-0312"))

counts_before = {n: db.collection(n).count() for n in (AG.documents, AG.chunks, AG.entities) if n}

In [ ]:
from arango_ai_sdk.exceptions import CategoryAlreadyBuiltError

new_doc = CORPUS_DIR / "incremental" / "AD-EASA-2026-0214.md"


def show_ingest(p):
    s = p.status
    detail = getattr(s, "phase", None) or getattr(s, "progress", None) or ""
    print(p.step, s.status, detail)


update_category = CATEGORY
print(client.files.upload_file(new_doc, scope=[PROJECT, update_category]))
try:
    update = project.autograph.ingest([update_category], incremental=True, on_progress=show_ingest)
except CategoryAlreadyBuiltError as exc:
    # Fallback if the service won't add to a built category: use a new category.
    print("same-category incremental rejected:", exc)
    update_category = f"{CATEGORY}_2026q4"
    print(client.files.upload_file(new_doc, scope=[PROJECT, update_category]))
    update = project.autograph.ingest([update_category], incremental=True, on_progress=show_ingest)

print("category used :", update_category)
print("added / unchanged / removed:", update.build.documents_added,
      update.build.documents_unchanged, update.build.documents_removed)
print("orchestration :", "nothing stale" if update.orchestration is None
      else update.orchestration.status)

In [ ]:
counts_after = {n: db.collection(n).count() for n in counts_before}
print({n: f"{counts_before[n]} → {counts_after[n]}" for n in counts_before})

# The new directive in the generated graph
peek(db, """
FOR c IN @@chunks
  FILTER CONTAINS(c.content, @doc)
  LIMIT 5
  RETURN {chunk: c._key, preview: SUBSTRING(c.content, 0, 160)}
""", {"@chunks": AG.chunks, "doc": "AD-EASA-2026-0214"})

In [ ]:
after = rag.ask(C1, query_type=QueryType.UNIFIED)
show(f"AFTER — {C1}", after)
print(answer_mentions(after, "AD-EASA-2026-0214", "50 cycles", "10 cycles", "stage 1-4"))

In [ ]:
C2 = ("Which CFM56-7B27 engines must be inspected within 10 cycles under AD-EASA-2026-0214, and "
      "can we replace the HPC blade set stage 1-4 if they fail?")
c2 = deep_search(C2)
show(f"[Deep Search, after update] {C2}", c2)
print(answer_mentions(c2, "HPC blade set", "stock"))

## 9 · Cleanup (optional)

Undeploys the retriever and AutoGraph services. The project, secret profiles, uploaded files and imported collections remain (the SDK cannot delete them yet). Set `CLEANUP = True` to run.

In [ ]:
CLEANUP = False
if CLEANUP:
    print("retriever undeployed:", rag.undeploy().service_id)
    print("autograph undeployed:", project.autograph.undeploy().service_id)
    client.close()